# **US Flights Delay:** Schema Engineering

## Imports

In [1]:
import sys

In [2]:
sys.path.append("../src/database")
sys.path.append("../src/database/queries")

In [3]:
import connection 
import base_queries
import query_executor

## Connect to client and database

In [4]:
MONGODB_URI = "mongodb://localhost:27017/"
MONGODB_NAME = "flights_delay_db"

In [5]:
client, database = connection.connect_to_database(uri = MONGODB_URI, db_name = MONGODB_NAME)

In [6]:
database.list_collection_names()

['airports',
 'airports_summary_hybrid_optimized',
 'runways',
 'cancelled_deverted_2023',
 'flights_hybrid_optimized',
 'us_flights_2023',
 'weather_hybrid_optimized',
 'airport_frequencies',
 'airports_geolocation',
 'us_flights_optimized',
 'weather_meteo_by_airport']

## Queries

### **Flight Delay Analysis:** Base Queries

In [8]:
base_queries = base_queries.get_base_queries()

In [7]:
collection = database["us_flights_2023"]

#### **Q1:** Which US states have the highest average delays by season (Winter, Spring, Summer, Fall)?

Data from the **`us_flights_2023`** and **`airports_geolocation`** collections are analyzed.

The season is determined based on the `flight_date` field:
* **Winter:** December–February
* **Spring:** March–May
* **Summer:** June–August
* **Fall:** September–November

The average delay is calculated as `avg(dep_delay)` for all flights from a given state (`state` from `airports_geolocation`) in a given season.

**Result:** List of US states with average delay by season, sorted in descending order of average delay.

In [8]:
base_query_1 = base_queries['query_1']

In [9]:
results, execution_time = query_executor.execute_query(collection, base_query_1, "Query 1")

Query 'Query 1' executed in 412.3152 seconds


#### **Q2:** Which airlines have the highest average delays on rainy days?

Collections **`us_flights_2023`** and **`weather_meteo_by_airport`** are used.

Precipitation is taken from the `prcp' field (mm).

**Significant precipitation**: days when `prcp > 5.0`.

Need to find average delay (`avg(dep_delay)`) by airline (`airline`) only for days with significant precipitation, based on weather conditions from `departure.airport_code`.

**Result:** Airlines with average delay on days with precipitation > 5 mm, sorted in descending order of value.

In [ ]:
# Index for weather_meteo_by_airport
database["weather_meteo_by_airport"].create_index([
    ("airport_id", 1),
    ("time", 1),
    ("prcp", 1)
])

# Index for us_flights_2023
database["us_flights_2023"].create_index([
    ("Dep_Airport", 1),
    ("FlightDate", 1),
    ("Airline", 1)
])

In [ ]:
base_query_2 = base_queries['query_2']

#### **Q3:** Which airports have the most canceled flights during bad weather?

Collections **`cancelled_deverted_2023`**, **`weather_meteo_by_airport`** and **`airports_geolocation`** are used.

Canceled flights are those with `cancelled = 1`.

**Bad weather conditions** are defined as:
* `prcp > 10 mm' *(heavy precipitation)*
* **or** `wspd > 15 m/s' *(strong wind)*

The query should match ($lookup) flights and weather data by `dep_airport` and `airport_id`.

**Result:** Airports with the highest number of canceled flights during bad weather, sorted in descending order of cancellations.

In [ ]:
# Index for cancelled_deverted_2023
database["cancelled_deverted_2023"].create_index([
    ("Cancelled", 1),
    ("Dep_Airport", 1),
    ("FlightDate", 1)
], name="cancelled_flights_match")
    
# Index for weather_meteo_by_airport  
database["weather_meteo_by_airport"].create_index([
    ("airport_id", 1),
    ("time", 1),
    ("prcp", 1),
    ("wspd", 1)
], name="weather_lookup_optimized")
    
database["weather_meteo_by_airport"].create_index([
    ("airport_id", 1),
    ("time", 1),
    ("wspd", 1) 
], name="weather_wind_lookup")
    
# Index for airports_geolocation
database["airports_geolocation"].create_index([
    ("IATA_CODE", 1)
], name="airport_geo_lookup")

In [ ]:
base_query_3 = base_queries['query_3']

#### **Q4:** Do airports with more diverse runways have lower average delays?

Collections **`airports`**, **`runways`**, and **`us_flights_2023`** are used.

For each airport, the following is calculated:
* **Runway Diversity Index (RDI)** = number of different values ​​of `surface` from the collection of `runways` per airport.
* **Average Delay** = average `dep_delay` from `us_flights_2023` per airport.

It is necessary to merge (`$lookup') all three collections, calculate both metrics, and analyze whether airports with higher RDI have lower average delays.

**Result:** List of airports with RDI and average delay, sorted in ascending order of average delay.

In [ ]:
base_query_4 = base_queries['query_4']

#### **Q5:** Which airlines are most affected by **weather-related delays** at **high-elevation airports** with **complex communication frequency environments**?

This query uses collections: **`us_flights_2023`**, **`airports`**, and **`airport_frequencies`**.

We analyze how **weather delays** vary across airlines **depending on characteristics of the airports they operate from** and return **top 10** airlines most affected. 

**Result:**  A ranked list of airlines operating in **high-elevation, high-complexity airports**, showing how strongly **weather delays** affect them. 

In [ ]:
base_query_5 = base_queries['query_5']

### **Testing of query execution**

In [30]:
import json

test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Airline" : { "$ne" : None }
        }
    },
    {
        "$lookup" : {
            "from" : "weather_meteo_by_airport",
            "let" : {
                "dep_airport" : "$Dep_Airport",
                "flight_date" : "$FlightDate"
            },
            "pipeline" : [
                {
                    "$match" : {
                        "$expr" : {
                            "$and" : [
                                { "$eq" : [ "$airport_id", "$$dep_airport"] },
                                { "$eq" : [ "$time", "$$flight_date"] },
                                { "$gt" : [ "$prcp", 5.0] },
                                { "$ne" : [ "$prcp", None] }
                            ]
                        }
                    }
                },
                {"$project" : {"_id" : 1, "prcp" : 1}}  # Return min data cuz we just need to know if it exists
            ],
            "as" : "weather_info"
        }
    },
    {
        "$group" : {
            "_id" : "$Airline",
            "average_delay" : { "$avg" : "$Dep_Delay" },
            "flight_count" : { "$sum" : 1 },
            "total_delay_min" : { "$sum" : "$Dep_Delay" }
        }
    },
    {
        "$sort" : { "average_delay" : -1}
    },
    {
        "$limit" : 5
    }
]

try:
    test_results = collection.aggregate(test_pipeline, allowDiskUse=True)
    test_results = list(test_results)
    
    print("Test Results:")
    for i, doc in enumerate(test_results, 1):
        print(f"{i}. Celokupan dokument:")
        print(json.dumps(doc, indent=2, default=str))
        print()
        
except Exception as e:
    print(f"Test Error: {e}")

Test Results:
1. Celokupan dokument:
{
  "_id": "JetBlue Airways",
  "average_delay": 24.27050743706026,
  "flight_count": 267915,
  "total_delay_min": 6502433
}

2. Celokupan dokument:
{
  "_id": "Frontier Airlines Inc.",
  "average_delay": 22.158256417943146,
  "flight_count": 173459,
  "total_delay_min": 3843549
}

3. Celokupan dokument:
{
  "_id": "Spirit Air Lines",
  "average_delay": 19.540724314049715,
  "flight_count": 258838,
  "total_delay_min": 5057882
}

4. Celokupan dokument:
{
  "_id": "American Airlines Inc.",
  "average_delay": 17.43667960407647,
  "flight_count": 928058,
  "total_delay_min": 16182250
}

5. Celokupan dokument:
{
  "_id": "Allegiant Air",
  "average_delay": 15.38555385623771,
  "flight_count": 114425,
  "total_delay_min": 1760492
}



#### **Q3:** Which airports have the most canceled flights during bad weather?

Collections **`cancelled_deverted_2023`**, **`weather_meteo_by_airport`** and **`airports_geolocation`** are used.

Canceled flights are those with `cancelled = 1`.

**Bad weather conditions** are defined as:
* `prcp > 10 mm' *(heavy precipitation)*
* **or** `wspd > 15 m/s' *(strong wind)*

The query should match ($lookup) flights and weather data by `dep_airport` and `airport_id`.

**Result:** Airports with the highest number of canceled flights during bad weather, sorted in descending order of cancellations.

In [33]:
collection = database["cancelled_deverted_2023"]

In [44]:
test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "Cancelled" : { "$eq" : 1 }
        }
    },
    {
        "$lookup" : {
            "from" : "weather_meteo_by_airport",
            "let" : {
                "dep_airport" : "$Dep_Airport",
                "flight_date" : "$FlightDate"
            },
            "pipeline" : [
                {
                    "$match" : {
                        "$expr" : {
                            "$and" : [
                                { "$eq" : [ "$airport_id", "$$dep_airport" ]},
                                { "$eq" : [ "$time", "$$flight_date" ]},
                                { "$or" : [
                                    { "$gt" : [ "$prcp", 5.0]},
                                    { "$gt" : [ "$wspd", 15]}
                                ]}
                            ]
                        }
                    }
                },
                {
                    "$project" : { "_id" : 1 }
                }
            ],
            "as" : "weather_info"
        }
    },
    {
        "$match": {
            "weather_info" : {"$ne" : [] }
        }
    },
    {
        "$lookup" : {
            "from" : "airports_geolocation",
            "localField" : "Dep_Airport",
            "foreignField" : "IATA_CODE",
            "as" : "airport_info"
        }
    },

    {
        "$group" : {
            "_id" : "$Dep_Airport",
            "cancelled_flights_count" : { "$sum" : 1 },
            "city" : { "$first" : "$airport_info.CITY" },
            "state" : { "$first" : "$airport_info.STATE" }
        }
    },
    { 
        "$sort" : { "cancelled_flights_count" : -1 }
    }
]

try:
    test_results = collection.aggregate(test_pipeline, allowDiskUse=True)
    test_results = list(test_results)
    
    print("Test Results:")
    for i, doc in enumerate(test_results, 1):
        print(f"{i}. Celokupan dokument:")
        print(json.dumps(doc, indent=2, default=str))
        print()
        
except Exception as e:
    print(f"Test Error: {e}")

Test Results:
1. Celokupan dokument:
{
  "_id": "DFW",
  "cancelled_flights_count": 3870,
  "city": [
    "Dallas-Fort Worth"
  ],
  "state": [
    "TX"
  ]
}

2. Celokupan dokument:
{
  "_id": "DEN",
  "cancelled_flights_count": 2568,
  "city": [
    "Denver"
  ],
  "state": [
    "CO"
  ]
}

3. Celokupan dokument:
{
  "_id": "LGA",
  "cancelled_flights_count": 2504,
  "city": [
    "New York"
  ],
  "state": [
    "NY"
  ]
}

4. Celokupan dokument:
{
  "_id": "JFK",
  "cancelled_flights_count": 2356,
  "city": [
    "New York"
  ],
  "state": [
    "NY"
  ]
}

5. Celokupan dokument:
{
  "_id": "EWR",
  "cancelled_flights_count": 2294,
  "city": [
    "Newark"
  ],
  "state": [
    "NJ"
  ]
}

6. Celokupan dokument:
{
  "_id": "ORD",
  "cancelled_flights_count": 2105,
  "city": [
    "Chicago"
  ],
  "state": [
    "IL"
  ]
}

7. Celokupan dokument:
{
  "_id": "BOS",
  "cancelled_flights_count": 1847,
  "city": [
    "Boston"
  ],
  "state": [
    "MA"
  ]
}

8. Celokupan dokument:
{


Part 2:
```
Test Results:
1. Celokupan dokument:
{
  "_id": "LGA",
  "cancelled_flights_count": 4486
}

2. Celokupan dokument:
{
  "_id": "DFW",
  "cancelled_flights_count": 4414
}

3. Celokupan dokument:
{
  "_id": "DEN",
  "cancelled_flights_count": 4026
}

4. Celokupan dokument:
{
  "_id": "EWR",
  "cancelled_flights_count": 3836
}

5. Celokupan dokument:
{
  "_id": "ORD",
  "cancelled_flights_count": 3152
}

```

In [ ]:
test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "Cancelled" : { "$eq" : 1 }
        }
    },
    {
        "$group" : {
            "_id" : "$Dep_Airport",
            "cancelled_flights_count" : { "$sum" : 1 }
        }
    },
    { 
        "$sort" : { "cancelled_flights_count" : -1 }
    },
    {
        "$limit" : 5
    }
]

Part 3:
```
Test Results:
1. Celokupan dokument:
{
  "_id": "LGA",
  "cancelled_flights_count": 4486,
  "city": [
    "New York"
  ],
  "state": [
    "NY"
  ]
}

2. Celokupan dokument:
{
  "_id": "DFW",
  "cancelled_flights_count": 4414,
  "city": [
    "Dallas-Fort Worth"
  ],
  "state": [
    "TX"
  ]
}

3. Celokupan dokument:
{
  "_id": "DEN",
  "cancelled_flights_count": 4026,
  "city": [
    "Denver"
  ],
  "state": [
    "CO"
  ]
}

4. Celokupan dokument:
{
  "_id": "EWR",
  "cancelled_flights_count": 3836,
  "city": [
    "Newark"
  ],
  "state": [
    "NJ"
  ]
}

5. Celokupan dokument:
{
  "_id": "ORD",
  "cancelled_flights_count": 3152,
  "city": [
    "Chicago"
  ],
  "state": [
    "IL"
  ]
}
```

In [ ]:
test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "Cancelled" : { "$eq" : 1 }
        }
    },
    {
        "$lookup" : {
            "from" : "airports_geolocation",
            "localField" : "Dep_Airport",
            "foreignField" : "IATA_CODE",
            "as" : "airport_info"
        }
    },
    {
        "$group" : {
            "_id" : "$Dep_Airport",
            "cancelled_flights_count" : { "$sum" : 1 },
            "city" : { "$first" : "$airport_info.CITY" },
            "state" : { "$first" : "$airport_info.STATE" }
        }
    },
    { 
        "$sort" : { "cancelled_flights_count" : -1 }
    },
    {
        "$limit" : 5
    }
]

```
Test Results:
1. Celokupan dokument:
{
  "_id": "DFW",
  "cancelled_flights_count": 3870,
  "city": [
    "Dallas-Fort Worth"
  ],
  "state": [
    "TX"
  ]
}

2. Celokupan dokument:
{
  "_id": "DEN",
  "cancelled_flights_count": 2568,
  "city": [
    "Denver"
  ],
  "state": [
    "CO"
  ]
}

3. Celokupan dokument:
{
  "_id": "LGA",
  "cancelled_flights_count": 2504,
  "city": [
    "New York"
  ],
  "state": [
    "NY"
  ]
}

4. Celokupan dokument:
{
  "_id": "JFK",
  "cancelled_flights_count": 2356,
  "city": [
    "New York"
  ],
  "state": [
    "NY"
  ]
}

5. Celokupan dokument:
{
  "_id": "EWR",
  "cancelled_flights_count": 2294,
  "city": [
    "Newark"
  ],
  "state": [
    "NJ"
  ]
}
```

In [ ]:
test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "Cancelled" : { "$eq" : 1 }
        }
    },
    {
        "$lookup" : {
            "from" : "weather_meteo_by_airport",
            "let" : {
                "dep_airport" : "$Dep_Airport",
                "flight_date" : "$FlightDate"
            },
            "pipeline" : [
                {
                    "$match" : {
                        "$expr" : {
                            "$and" : [
                                { "$eq" : [ "$airport_id", "$$dep_airport" ]},
                                { "$eq" : [ "$time", "$$flight_date" ]},
                                { "$or" : [
                                    { "$gt" : [ "$prcp", 5.0]},
                                    { "$gt" : [ "$wspd", 15]}
                                ]}
                            ]
                        }
                    }
                },
                {
                    "$project" : { "_id" : 1 }
                }
            ],
            "as" : "weather_info"
        }
    },
    {
        "$match": {
            "weather_info" : {"$ne" : [] }
        }
    },
    {
        "$lookup" : {
            "from" : "airports_geolocation",
            "localField" : "Dep_Airport",
            "foreignField" : "IATA_CODE",
            "as" : "airport_info"
        }
    },

    {
        "$group" : {
            "_id" : "$Dep_Airport",
            "cancelled_flights_count" : { "$sum" : 1 },
            "city" : { "$first" : "$airport_info.CITY" },
            "state" : { "$first" : "$airport_info.STATE" }
        }
    },
    { 
        "$sort" : { "cancelled_flights_count" : -1 }
    }
]


**Q4:** Do airports with more diverse runways have lower average delays?

Collections **`airports`**, **`runways`**, and **`us_flights_2023`** are used.

For each airport, the following is calculated:
* **Runway Diversity Index (RDI)** = number of different values ​​of `surface` from the collection of `runways` per airport.
* **Average Delay** = average `dep_delay` from `us_flights_2023` per airport.

It is necessary to merge (`$lookup') all three collections, calculate both metrics, and analyze whether airports with higher RDI have lower average delays.

**Result:** List of airports with RDI and average delay, sorted in ascending order of average delay.

In [49]:
collection = database["airports"]

In [57]:
test_pipeline = [
    {
        "$match" : {
            "iata_code" : { "$ne" : None },
            "type" : { "$in" : ["medium_airport", "large_airport"]}
        }
    },
    {
        "$lookup" : {
            "from" : "runways",
            "localField" : "ident",
            "foreignField" : "airport_ident",
            "as" : "runway_info"
        },
    },
    {
        "$lookup" : {
            "from" : "us_flights_2023",
            "let" : {
                "airport_code" : "$iata_code"
            },
            "pipeline" : [
                {
                    "$match" : {
                        "$expr" : {
                            "$eq" : [ "$Dep_Airport", "$$airport_code" ]
                        }
                    }
                },
                {
                    "$group" : {
                        "_id" : "$Dep_Airport",
                        "average_delay" : { "$avg" : "$Dep_Delay" },
                        "total_flights" : { "$sum" : 1 }
                    }
                }
            ],
            "as" : "flight_info"
        }
    },
    {
        "$match" : {
            "runway_info" : { "$ne" : [] },
            "flight_info" : { "$ne" : [] }
        }
    },
    {
        "$addFields" : { 
            "RDI": { 
                "$size": { 
                    "$setUnion": [
                        { "$map": { 
                            "input": "$runway_info", 
                            "as": "runway", 
                            "in": "$$runway.surface"
                        }}, 
                        [] 
                    ] 
                } 
            }
        }
    },
    {
        "$project" : {
            "airport_code" : "$iata_code",
            "airport_name" : "$name",
            "RDI" : 1,
            "average_delay" : {
                "$round" : [ { "$arrayElemAt": ["$flight_info.average_delay", 0] }, 2] 
            },
            "total_flights": { 
                "$arrayElemAt": ["$flight_info.total_flights", 0] 
            },
            "runway_surfaces": {
                "$map": {
                    "input": "$runway_info",
                    "as": "runway",
                    "in": "$$runway.surface"
                }
            }
        }
    },
    {
        "$sort" : { "average_delay" : -1, "total_flights" : -1 }
    }
]

try:
    test_results = collection.aggregate(test_pipeline, allowDiskUse = True)
    test_results = list(test_results)

    print("Test Results:")
    for i, doc in enumerate(test_results, 1):
        print(f"{i}. Ceo dokument:")
        print(json.dumps(doc, indent=2, default=str))
        print()
except Exception as e:
    print(f"Test Error: {e}")

Test Results:
1. Ceo dokument:
{
  "_id": "68f2e8a7bb35bab1fe210a45",
  "RDI": 1,
  "airport_code": "SMX",
  "airport_name": "Santa Maria Pub/Capt G Allan Hancock Field",
  "average_delay": 61.54,
  "total_flights": 106,
  "runway_surfaces": [
    "ASP",
    "ASP"
  ]
}

2. Ceo dokument:
{
  "_id": "68f2e8a7bb35bab1fe2103dc",
  "RDI": 1,
  "airport_code": "COD",
  "airport_name": "Yellowstone Regional Airport",
  "average_delay": 46.97,
  "total_flights": 33,
  "runway_surfaces": [
    "ASP"
  ]
}

3. Ceo dokument:
{
  "_id": "68f2e8a7bb35bab1fe210644",
  "RDI": 1,
  "airport_code": "HTS",
  "airport_name": "Tri-State/Milton J. Ferguson Field",
  "average_delay": 34.01,
  "total_flights": 397,
  "runway_surfaces": [
    "ASP",
    "ASP"
  ]
}

4. Ceo dokument:
{
  "_id": "68f2e8a7bb35bab1fe21031a",
  "RDI": 1,
  "airport_code": "BIH",
  "airport_name": "Eastern Sierra Regional Airport",
  "average_delay": 29.34,
  "total_flights": 241,
  "runway_surfaces": [
    "ASP",
    "ASP",
    "

**Part 3:**
```
Test Results:
1. Ceo dokument:
{
  "_id": "68f2e8a7bb35bab1fe21024c",
  "id": 3356,
  "ident": "KABE",
  "type": "medium_airport",
  "name": "Lehigh Valley International Airport",
  "latitude_deg": 40.652099609375,
  "longitude_deg": -75.44080352783203,
  "elevation_ft": 393.0,
  "iso_country": "US",
  "iso_region": "US-PA",
  "municipality": "Allentown",
  "scheduled_service": "yes",
  "gps_code": "KABE",
  "iata_code": "ABE",
  "local_code": "ABE",
  "wikipedia_link": "https://en.wikipedia.org/wiki/Lehigh_Valley_International_Airport",
  "runway_info": [
    {
      "_id": "68f2e89bbb35bab1fe2039b7",
      "id": 240632,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "length_ft": 7600,
      "width_ft": 150,
      "surface": "ASP",
      "lighted": 1,
      "closed": 0,
      "le_ident": 6,
      "le_latitude_deg": 40.647,
      "le_longitude_deg": -75.4506,
      "le_elevation_ft": 394,
      "le_heading_degT": 51.3,
      "he_ident": 24,
      "he_latitude_deg": 40.66,
      "he_longitude_deg": -75.4293,
      "he_elevation_ft": 380,
      "he_heading_degT": 231.3,
      "he_displaced_threshold_ft": 500
    },
    {
      "_id": "68f2e89bbb35bab1fe2039b8",
      "id": 240633,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "length_ft": 5797,
      "width_ft": 150,
      "surface": "ASP",
      "lighted": 1,
      "closed": 0,
      "le_ident": 13,
      "le_latitude_deg": 40.6552,
      "le_longitude_deg": -75.4498,
      "le_elevation_ft": 383,
      "le_heading_degT": 123.1,
      "he_ident": 31,
      "he_latitude_deg": 40.6465,
      "he_longitude_deg": -75.4322,
      "he_elevation_ft": 380,
      "he_heading_degT": 303.1
    }
  ],
  "flight_info": [
    {
      "_id": "ABE",
      "average_delay": 9.269001831501832,
      "total_flights": 4368
    }
  ]
}

2. Ceo dokument:
{
  "_id": "68f2e8a7bb35bab1fe21024d",
  "id": 3357,
  "ident": "KABI",
  "type": "medium_airport",
  "name": "Abilene Regional Airport",
  "latitude_deg": 32.4113006592,
  "longitude_deg": -99.6819000244,
  "elevation_ft": 1791.0,
  "iso_country": "US",
  "iso_region": "US-TX",
  "municipality": "Abilene",
  "scheduled_service": "yes",
  "gps_code": "KABI",
  "iata_code": "ABI",
  "local_code": "ABI",
  "wikipedia_link": "https://en.wikipedia.org/wiki/Abilene_Regional_Airport",
  "runway_info": [
    {
      "_id": "68f2e89bbb35bab1fe2039b9",
      "id": 243999,
      "airport_ref": 3357,
      "airport_ident": "KABI",
      "length_ft": 3678,
      "width_ft": 100,
      "surface": "ASP",
      "lighted": 1,
      "closed": 0,
      "le_ident": 4,
      "le_latitude_deg": 32.4196,
      "le_longitude_deg": -99.6948,
      "le_elevation_ft": 1751,
      "le_heading_degT": 52,
      "he_ident": 22,
      "he_latitude_deg": 32.4258,
      "he_longitude_deg": -99.6854,
      "he_elevation_ft": 1762,
      "he_heading_degT": 232
    },
    {
      "_id": "68f2e89bbb35bab1fe2039ba",
      "id": 244001,
      "airport_ref": 3357,
      "airport_ident": "KABI",
      "length_ft": 7198,
      "width_ft": 150,
      "surface": "ASP",
      "lighted": 1,
      "closed": 0,
      "le_ident": "17L",
      "le_latitude_deg": 32.4085,
      "le_longitude_deg": -99.6748,
      "le_elevation_ft": 1791,
      "le_heading_degT": 180,
      "he_ident": "35R",
      "he_latitude_deg": 32.3887,
      "he_longitude_deg": -99.6747,
      "he_elevation_ft": 1775,
      "he_heading_degT": 360
    },
    {
      "_id": "68f2e89bbb35bab1fe2039bb",
      "id": 244000,
      "airport_ref": 3357,
      "airport_ident": "KABI",
      "length_ft": 7202,
      "width_ft": 150,
      "surface": "ASP",
      "lighted": 1,
      "closed": 0,
      "le_ident": "17R",
      "le_latitude_deg": 32.4281,
      "le_longitude_deg": -99.6849,
      "le_elevation_ft": 1757,
      "le_heading_degT": 180,
      "he_ident": "35L",
      "he_latitude_deg": 32.4083,
      "he_longitude_deg": -99.6848,
      "he_elevation_ft": 1785,
      "he_heading_degT": 360
    }
  ],
  "flight_info": [
    {
      "_id": "ABI",
      "average_delay": 6.342281879194631,
      "total_flights": 1341
    }
  ]
}

3. Ceo dokument:
{
  "_id": "68f2e8a7bb35bab1fe21024e",
  "id": 16091,
  "ident": "KABQ",
  "type": "large_airport",
  "name": "Albuquerque International Sunport",
  "latitude_deg": 35.040199,
  "longitude_deg": -106.609001,
  "elevation_ft": 5355.0,
  "iso_country": "US",
  "iso_region": "US-NM",
  "municipality": "Albuquerque",
  "scheduled_service": "yes",
  "gps_code": "KABQ",
  "iata_code": "ABQ",
  "local_code": "ABQ",
  "home_link": "http://www.abqsunport.com/",
  "wikipedia_link": "https://en.wikipedia.org/wiki/Albuquerque_International_Sunport",
  "runway_info": [
    {
      "_id": "68f2e89bbb35bab1fe2039bc",
      "id": 253207,
      "airport_ref": 16091,
      "airport_ident": "KABQ",
      "length_ft": 10000,
      "width_ft": 150,
      "surface": "CONC-G",
      "lighted": 0,
      "closed": 0,
      "le_ident": 3,
      "le_latitude_deg": 35.0222,
      "le_longitude_deg": -106.631,
      "le_elevation_ft": 5305,
      "le_heading_degT": 45,
      "he_ident": 21,
      "he_latitude_deg": 35.0417,
      "he_longitude_deg": -106.607,
      "he_elevation_ft": 5316,
      "he_heading_degT": 225
    },
    {
      "_id": "68f2e89bbb35bab1fe2039bd",
      "id": 253208,
      "airport_ref": 16091,
      "airport_ident": "KABQ",
      "length_ft": 13793,
      "width_ft": 150,
      "surface": "CONC-G",
      "lighted": 0,
      "closed": 0,
      "le_ident": 8,
      "le_latitude_deg": 35.0443,
      "le_longitude_deg": -106.622,
      "le_elevation_ft": 5315,
      "le_heading_degT": 90,
      "he_ident": 26,
      "he_latitude_deg": 35.0441,
      "he_longitude_deg": -106.576,
      "he_elevation_ft": 5355,
      "he_heading_degT": 270
    },
    {
      "_id": "68f2e89bbb35bab1fe2039be",
      "id": 253209,
      "airport_ref": 16091,
      "airport_ident": "KABQ",
      "length_ft": 6000,
      "width_ft": 150,
      "surface": "CONC-G",
      "lighted": 0,
      "closed": 0,
      "le_ident": 12,
      "le_latitude_deg": 35.0435,
      "le_longitude_deg": -106.621,
      "le_elevation_ft": 5312,
      "le_heading_degT": 129,
      "he_ident": 30,
      "he_latitude_deg": 35.0332,
      "he_longitude_deg": -106.605,
      "he_elevation_ft": 5314,
      "he_heading_degT": 309
    },
    {
      "_id": "68f2e89bbb35bab1fe2039bf",
      "id": 253210,
      "airport_ref": 16091,
      "airport_ident": "KABQ",
      "length_ft": 10000,
      "width_ft": 150,
      "surface": "ASPH-CONC-F",
      "lighted": 0,
      "closed": 0,
      "le_ident": 17,
      "le_latitude_deg": 35.0577,
      "le_longitude_deg": -106.611,
      "le_elevation_ft": 5321,
      "le_heading_degT": 183,
      "he_ident": 35,
      "he_latitude_deg": 35.0303,
      "he_longitude_deg": -106.613,
      "he_elevation_ft": 5314,
      "he_heading_degT": 3
    }
  ],
  "flight_info": [
    {
      "_id": "ABQ",
      "average_delay": 9.981867399991406,
      "total_flights": 23273
    }
  ]
}

4. Ceo dokument:
{
  "_id": "68f2e8a7bb35bab1fe21024f",
  "id": 3358,
  "ident": "KABR",
  "type": "medium_airport",
  "name": "Aberdeen Regional Airport",
  "latitude_deg": 45.449100494384766,
  "longitude_deg": -98.42179870605467,
  "elevation_ft": 1302.0,
  "iso_country": "US",
  "iso_region": "US-SD",
  "municipality": "Aberdeen",
  "scheduled_service": "yes",
  "gps_code": "KABR",
  "iata_code": "ABR",
  "local_code": "ABR",
  "wikipedia_link": "https://en.wikipedia.org/wiki/Aberdeen_Regional_Airport",
  "runway_info": [
    {
      "_id": "68f2e89bbb35bab1fe2039c0",
      "id": 241085,
      "airport_ref": 3358,
      "airport_ident": "KABR",
      "length_ft": 6901,
      "width_ft": 100,
      "surface": "CON",
      "lighted": 1,
      "closed": 0,
      "le_ident": 13,
      "le_latitude_deg": 45.4552,
      "le_longitude_deg": -98.428,
      "le_elevation_ft": 1302,
      "le_heading_degT": 135.1,
      "he_ident": 31,
      "he_latitude_deg": 45.4418,
      "he_longitude_deg": -98.409,
      "he_elevation_ft": 1301,
      "he_heading_degT": 315.1
    },
    {
      "_id": "68f2e89bbb35bab1fe2039c1",
      "id": 241086,
      "airport_ref": 3358,
      "airport_ident": "KABR",
      "length_ft": 5500,
      "width_ft": 100,
      "surface": "ASP",
      "lighted": 1,
      "closed": 0,
      "le_ident": 17,
      "le_latitude_deg": 45.4556,
      "le_longitude_deg": -98.4274,
      "le_elevation_ft": 1302,
      "le_heading_degT": 180,
      "he_ident": 35,
      "he_latitude_deg": 45.4405,
      "he_longitude_deg": -98.4273,
      "he_elevation_ft": 1298,
      "he_heading_degT": 360
    }
  ],
  "flight_info": [
    {
      "_id": "ABR",
      "average_delay": 6.50773558368495,
      "total_flights": 711
    }
  ]
}

5. Ceo dokument:
{
  "_id": "68f2e8a7bb35bab1fe210250",
  "id": 3359,
  "ident": "KABY",
  "type": "medium_airport",
  "name": "Southwest Georgia Regional Airport",
  "latitude_deg": 31.535499572753903,
  "longitude_deg": -84.19450378417969,
  "elevation_ft": 197.0,
  "iso_country": "US",
  "iso_region": "US-GA",
  "municipality": "Albany",
  "scheduled_service": "yes",
  "gps_code": "KABY",
  "iata_code": "ABY",
  "local_code": "ABY",
  "wikipedia_link": "https://en.wikipedia.org/wiki/Southwest_Georgia_Regional_Airport",
  "runway_info": [
    {
      "_id": "68f2e89bbb35bab1fe2039c2",
      "id": 244793,
      "airport_ref": 3359,
      "airport_ident": "KABY",
      "length_ft": 6601,
      "width_ft": 150,
      "surface": "ASP",
      "lighted": 1,
      "closed": 0,
      "le_ident": 4,
      "le_latitude_deg": 31.5296,
      "le_longitude_deg": -84.1997,
      "le_elevation_ft": 196,
      "le_heading_degT": 42,
      "he_ident": 22,
      "he_latitude_deg": 31.5431,
      "he_longitude_deg": -84.1855,
      "he_elevation_ft": 189,
      "he_heading_degT": 222
    },
    {
      "_id": "68f2e89bbb35bab1fe2039c3",
      "id": 244794,
      "airport_ref": 3359,
      "airport_ident": "KABY",
      "length_ft": 5200,
      "width_ft": 150,
      "surface": "ASP",
      "lighted": 1,
      "closed": 0,
      "le_ident": 16,
      "le_latitude_deg": 31.5413,
      "le_longitude_deg": -84.1994,
      "le_elevation_ft": 195,
      "le_heading_degT": 162,
      "he_ident": 34,
      "he_latitude_deg": 31.5277,
      "he_longitude_deg": -84.1942,
      "he_elevation_ft": 196,
      "he_heading_degT": 342
    }
  ],
  "flight_info": [
    {
      "_id": "ABY",
      "average_delay": 7.291358024691358,
      "total_flights": 810
    }
  ]
}
```

**Q5:** Which airlines are most affected by **weather-related delays** at **high-elevation airports** with **complex communication frequency environments**?

This query uses collections: **`us_flights_2023`**, **`airports`**, and **`airport_frequencies`**.

Which airlines have the most weather-related delays (weather delay > 10 minutes), for flights from airports with an altitude of more than 500 feet and more than 5 communication frequencies?

We analyze how **weather delays** vary across airlines **depending on characteristics of the airports they operate from** and return **top 10** airlines most affected. 

**Result:**  A ranked list of airlines operating in **high-elevation, high-complexity airports**, showing how strongly **weather delays** affect them. 

In [7]:
collection = database["us_flights_2023"]

In [9]:
import json

In [16]:
test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "Airline" : { "$ne" : None },
            "Delay_Weather" : { "$gt" : 10, "$ne" : None }
        }
    },
    {
        "$group" : {
            "_id" : {
                "airline" : "$Airline",
                "airport" : "$Dep_Airport"
            },
            "weather_delay_count" : { "$sum" : 1 },
            "total_weather_delay" : { "$sum" : "$Delay_Weather" },
            "average_weather_delay" : { "$avg" : "$Delay_Weather" }
        }
    },
    {
        "$lookup" : {
            "from" : "airports",
            "localField" : "_id.airport",
            "foreignField" : "iata_code",
            "as" : "airport_info"
        }
    },
    {
        "$match" : {
            "airport_info" : { "$ne" : [] }
        }
    },
    {
        "$lookup" : {
            "from" : "airport_frequencies",
            "localField" : "airport_info.ident",
            "foreignField" : "airport_ident",
            "as" : "airport_freq_info"
        }
    },
    {
        "$match" : { 
            "airport_freq_info" : { "$ne" : [] } 
        }
    }
]

try:
    test_results = collection.aggregate(test_pipeline, allowDiskUse = True)
    test_results = list(test_results)

    print("Test Results:")
    for i, doc in enumerate(test_results, 1):
        print(f"{i}. Ceo dokument:")
        print(json.dumps(doc, indent=2, default=str))
        print()
except Exception as e:
    print(f"Test Error: {e}")

Test Results:
1. Ceo dokument:
{
  "_id": {
    "airline": "Alaska Airlines Inc.",
    "airport": "RSW"
  },
  "weather_delay_count": 1,
  "total_weather_delay": 34,
  "average_weather_delay": 34.0,
  "airport_info": [
    {
      "_id": "68f2e8a7bb35bab1fe2109ca",
      "id": 3858,
      "ident": "KRSW",
      "type": "large_airport",
      "name": "Southwest Florida International Airport",
      "latitude_deg": 26.53619956970215,
      "longitude_deg": -81.75520324707031,
      "elevation_ft": 30.0,
      "iso_country": "US",
      "iso_region": "US-FL",
      "municipality": "Fort Myers",
      "scheduled_service": "yes",
      "gps_code": "KRSW",
      "iata_code": "RSW",
      "local_code": "RSW",
      "home_link": "http://www.flylcpa.com/",
      "wikipedia_link": "https://en.wikipedia.org/wiki/Southwest_Florida_International_Airport"
    }
  ],
  "airport_freq_info": [
    {
      "_id": "68f2e8a2bb35bab1fe20cc89",
      "id": 69192,
      "airport_ref": 3858,
      "airport_id

**Part 1:**
```
Test Results:
1. Ceo dokument:
{
  "_id": "68f2e543bb35bab1feba72f2",
  "FlightDate": "2023-01-01 00:00:00",
  "Day_Of_Week": 7,
  "Airline": "Allegiant Air",
  "Tail_Number": "N255NV",
  "Dep_Airport": "ABE",
  "Dep_CityName": "Allentown/Bethlehem/Easton, PA",
  "DepTime_label": "Morning",
  "Dep_Delay": 21,
  "Dep_Delay_Tag": 1,
  "Dep_Delay_Type": "Medium >15min",
  "Arr_Airport": "PIE",
  "Arr_CityName": "St. Petersburg, FL",
  "Arr_Delay": 28,
  "Arr_Delay_Type": "Medium >15min",
  "Flight_Duration": 169,
  "Distance_type": "Short Haul >1500Mi",
  "Delay_Carrier": 0,
  "Delay_Weather": 21,
  "Delay_NAS": 7,
  "Delay_Security": 0,
  "Delay_LastAircraft": 0,
  "Manufacturer": "AIRBUS",
  "Model": "A320",
  "Aicraft_age": 7,
  "airport_info": [
    {
      "_id": "68f2e8a7bb35bab1fe21024c",
      "id": 3356,
      "ident": "KABE",
      "type": "medium_airport",
      "name": "Lehigh Valley International Airport",
      "latitude_deg": 40.652099609375,
      "longitude_deg": -75.44080352783203,
      "elevation_ft": 393.0,
      "iso_country": "US",
      "iso_region": "US-PA",
      "municipality": "Allentown",
      "scheduled_service": "yes",
      "gps_code": "KABE",
      "iata_code": "ABE",
      "local_code": "ABE",
      "wikipedia_link": "https://en.wikipedia.org/wiki/Lehigh_Valley_International_Airport"
    }
  ]
}

2. Ceo dokument:
{
  "_id": "68f2e550bb35bab1febe2006",
  "FlightDate": "2023-01-09 00:00:00",
  "Day_Of_Week": 1,
  "Airline": "Skywest Airlines Inc.",
  "Tail_Number": "N679CA",
  "Dep_Airport": "ABE",
  "Dep_CityName": "Allentown/Bethlehem/Easton, PA",
  "DepTime_label": "Afternoon",
  "Dep_Delay": 157,
  "Dep_Delay_Tag": 1,
  "Dep_Delay_Type": "Hight >60min",
  "Arr_Airport": "ATL",
  "Arr_CityName": "Atlanta, GA",
  "Arr_Delay": 133,
  "Arr_Delay_Type": "Hight >60min",
  "Flight_Duration": 118,
  "Distance_type": "Short Haul >1500Mi",
  "Delay_Carrier": 0,
  "Delay_Weather": 133,
  "Delay_NAS": 0,
  "Delay_Security": 0,
  "Delay_LastAircraft": 0,
  "Manufacturer": "CANADAIR REGIONAL JET",
  "Model": "CRJ",
  "Aicraft_age": 17,
  "airport_info": [
    {
      "_id": "68f2e8a7bb35bab1fe21024c",
      "id": 3356,
      "ident": "KABE",
      "type": "medium_airport",
      "name": "Lehigh Valley International Airport",
      "latitude_deg": 40.652099609375,
      "longitude_deg": -75.44080352783203,
      "elevation_ft": 393.0,
      "iso_country": "US",
      "iso_region": "US-PA",
      "municipality": "Allentown",
      "scheduled_service": "yes",
      "gps_code": "KABE",
      "iata_code": "ABE",
      "local_code": "ABE",
      "wikipedia_link": "https://en.wikipedia.org/wiki/Lehigh_Valley_International_Airport"
    }
  ]
}

3. Ceo dokument:
{
  "_id": "68f2e550bb35bab1febe2c98",
  "FlightDate": "2023-01-12 00:00:00",
  "Day_Of_Week": 4,
  "Airline": "Skywest Airlines Inc.",
  "Tail_Number": "N897SK",
  "Dep_Airport": "ABE",
  "Dep_CityName": "Allentown/Bethlehem/Easton, PA",
  "DepTime_label": "Afternoon",
  "Dep_Delay": 19,
  "Dep_Delay_Tag": 1,
  "Dep_Delay_Type": "Medium >15min",
  "Arr_Airport": "ATL",
  "Arr_CityName": "Atlanta, GA",
  "Arr_Delay": 20,
  "Arr_Delay_Type": "Medium >15min",
  "Flight_Duration": 146,
  "Distance_type": "Short Haul >1500Mi",
  "Delay_Carrier": 0,
  "Delay_Weather": 20,
  "Delay_NAS": 0,
  "Delay_Security": 0,
  "Delay_LastAircraft": 0,
  "Manufacturer": "CANADAIR REGIONAL JET",
  "Model": "CRJ",
  "Aicraft_age": 17,
  "airport_info": [
    {
      "_id": "68f2e8a7bb35bab1fe21024c",
      "id": 3356,
      "ident": "KABE",
      "type": "medium_airport",
      "name": "Lehigh Valley International Airport",
      "latitude_deg": 40.652099609375,
      "longitude_deg": -75.44080352783203,
      "elevation_ft": 393.0,
      "iso_country": "US",
      "iso_region": "US-PA",
      "municipality": "Allentown",
      "scheduled_service": "yes",
      "gps_code": "KABE",
      "iata_code": "ABE",
      "local_code": "ABE",
      "wikipedia_link": "https://en.wikipedia.org/wiki/Lehigh_Valley_International_Airport"
    }
  ]
}

4. Ceo dokument:
{
  "_id": "68f2e550bb35bab1febdf4bd",
  "FlightDate": "2023-01-22 00:00:00",
  "Day_Of_Week": 7,
  "Airline": "Skywest Airlines Inc.",
  "Tail_Number": "N549CA",
  "Dep_Airport": "ABE",
  "Dep_CityName": "Allentown/Bethlehem/Easton, PA",
  "DepTime_label": "Afternoon",
  "Dep_Delay": 3,
  "Dep_Delay_Tag": 1,
  "Dep_Delay_Type": "Low <5min",
  "Arr_Airport": "ATL",
  "Arr_CityName": "Atlanta, GA",
  "Arr_Delay": 25,
  "Arr_Delay_Type": "Medium >15min",
  "Flight_Duration": 167,
  "Distance_type": "Short Haul >1500Mi",
  "Delay_Carrier": 0,
  "Delay_Weather": 25,
  "Delay_NAS": 0,
  "Delay_Security": 0,
  "Delay_LastAircraft": 0,
  "Manufacturer": "CANADAIR REGIONAL JET",
  "Model": "CRJ",
  "Aicraft_age": 16,
  "airport_info": [
    {
      "_id": "68f2e8a7bb35bab1fe21024c",
      "id": 3356,
      "ident": "KABE",
      "type": "medium_airport",
      "name": "Lehigh Valley International Airport",
      "latitude_deg": 40.652099609375,
      "longitude_deg": -75.44080352783203,
      "elevation_ft": 393.0,
      "iso_country": "US",
      "iso_region": "US-PA",
      "municipality": "Allentown",
      "scheduled_service": "yes",
      "gps_code": "KABE",
      "iata_code": "ABE",
      "local_code": "ABE",
      "wikipedia_link": "https://en.wikipedia.org/wiki/Lehigh_Valley_International_Airport"
    }
  ]
}

5. Ceo dokument:
{
  "_id": "68f2e550bb35bab1febdee79",
  "FlightDate": "2023-01-23 00:00:00",
  "Day_Of_Week": 1,
  "Airline": "Skywest Airlines Inc.",
  "Tail_Number": "N800SK",
  "Dep_Airport": "ABE",
  "Dep_CityName": "Allentown/Bethlehem/Easton, PA",
  "DepTime_label": "Afternoon",
  "Dep_Delay": 34,
  "Dep_Delay_Tag": 1,
  "Dep_Delay_Type": "Medium >15min",
  "Arr_Airport": "ATL",
  "Arr_CityName": "Atlanta, GA",
  "Arr_Delay": 27,
  "Arr_Delay_Type": "Medium >15min",
  "Flight_Duration": 135,
  "Distance_type": "Short Haul >1500Mi",
  "Delay_Carrier": 0,
  "Delay_Weather": 27,
  "Delay_NAS": 0,
  "Delay_Security": 0,
  "Delay_LastAircraft": 0,
  "Manufacturer": "CANADAIR REGIONAL JET",
  "Model": "CRJ",
  "Aicraft_age": 18,
  "airport_info": [
    {
      "_id": "68f2e8a7bb35bab1fe21024c",
      "id": 3356,
      "ident": "KABE",
      "type": "medium_airport",
      "name": "Lehigh Valley International Airport",
      "latitude_deg": 40.652099609375,
      "longitude_deg": -75.44080352783203,
      "elevation_ft": 393.0,
      "iso_country": "US",
      "iso_region": "US-PA",
      "municipality": "Allentown",
      "scheduled_service": "yes",
      "gps_code": "KABE",
      "iata_code": "ABE",
      "local_code": "ABE",
      "wikipedia_link": "https://en.wikipedia.org/wiki/Lehigh_Valley_International_Airport"
    }
  ]
}
```

In [ ]:
test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "Airline" : { "$ne" : None },
            "Delay_Weather" : { "$gt" : 10, "$ne" : None }
        }
    },
    {
        "$lookup" : {
            "from" : "airports",
            "localField" : "Dep_Airport",
            "foreignField" : "iata_code",
            "as" : "airport_info"
        }
    },
    {
        "$limit" : 5
    }
]

**Part 2:**
```
Test Results:
1. Ceo dokument:
{
  "_id": "68f2e543bb35bab1feba72f2",
  "FlightDate": "2023-01-01 00:00:00",
  "Day_Of_Week": 7,
  "Airline": "Allegiant Air",
  "Tail_Number": "N255NV",
  "Dep_Airport": "ABE",
  "Dep_CityName": "Allentown/Bethlehem/Easton, PA",
  "DepTime_label": "Morning",
  "Dep_Delay": 21,
  "Dep_Delay_Tag": 1,
  "Dep_Delay_Type": "Medium >15min",
  "Arr_Airport": "PIE",
  "Arr_CityName": "St. Petersburg, FL",
  "Arr_Delay": 28,
  "Arr_Delay_Type": "Medium >15min",
  "Flight_Duration": 169,
  "Distance_type": "Short Haul >1500Mi",
  "Delay_Carrier": 0,
  "Delay_Weather": 21,
  "Delay_NAS": 7,
  "Delay_Security": 0,
  "Delay_LastAircraft": 0,
  "Manufacturer": "AIRBUS",
  "Model": "A320",
  "Aicraft_age": 7,
  "airport_info": [
    {
      "_id": "68f2e8a7bb35bab1fe21024c",
      "id": 3356,
      "ident": "KABE",
      "type": "medium_airport",
      "name": "Lehigh Valley International Airport",
      "latitude_deg": 40.652099609375,
      "longitude_deg": -75.44080352783203,
      "elevation_ft": 393.0,
      "iso_country": "US",
      "iso_region": "US-PA",
      "municipality": "Allentown",
      "scheduled_service": "yes",
      "gps_code": "KABE",
      "iata_code": "ABE",
      "local_code": "ABE",
      "wikipedia_link": "https://en.wikipedia.org/wiki/Lehigh_Valley_International_Airport"
    }
  ],
  "airport_freq_info": [
    {
      "_id": "68f2e8a2bb35bab1fe20aa51",
      "id": 60281,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "A/D",
      "description": "APP/DEP",
      "frequency_mhz": 118.2
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa52",
      "id": 60282,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "ATIS",
      "description": "ATIS",
      "frequency_mhz": 126.975
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa53",
      "id": 60283,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "CLD",
      "description": "CLNC DEL",
      "frequency_mhz": 124.05
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa54",
      "id": 60284,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "GND",
      "description": "GND",
      "frequency_mhz": 121.9
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa55",
      "id": 60285,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "RDO",
      "description": "WILLIAMSPORT RDO",
      "frequency_mhz": 117.5
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa56",
      "id": 60286,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "TWR",
      "description": "TWR",
      "frequency_mhz": 120.5
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa57",
      "id": 60287,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "UNIC",
      "description": "UNICOM",
      "frequency_mhz": 122.95
    }
  ]
}

2. Ceo dokument:
{
  "_id": "68f2e550bb35bab1febe2006",
  "FlightDate": "2023-01-09 00:00:00",
  "Day_Of_Week": 1,
  "Airline": "Skywest Airlines Inc.",
  "Tail_Number": "N679CA",
  "Dep_Airport": "ABE",
  "Dep_CityName": "Allentown/Bethlehem/Easton, PA",
  "DepTime_label": "Afternoon",
  "Dep_Delay": 157,
  "Dep_Delay_Tag": 1,
  "Dep_Delay_Type": "Hight >60min",
  "Arr_Airport": "ATL",
  "Arr_CityName": "Atlanta, GA",
  "Arr_Delay": 133,
  "Arr_Delay_Type": "Hight >60min",
  "Flight_Duration": 118,
  "Distance_type": "Short Haul >1500Mi",
  "Delay_Carrier": 0,
  "Delay_Weather": 133,
  "Delay_NAS": 0,
  "Delay_Security": 0,
  "Delay_LastAircraft": 0,
  "Manufacturer": "CANADAIR REGIONAL JET",
  "Model": "CRJ",
  "Aicraft_age": 17,
  "airport_info": [
    {
      "_id": "68f2e8a7bb35bab1fe21024c",
      "id": 3356,
      "ident": "KABE",
      "type": "medium_airport",
      "name": "Lehigh Valley International Airport",
      "latitude_deg": 40.652099609375,
      "longitude_deg": -75.44080352783203,
      "elevation_ft": 393.0,
      "iso_country": "US",
      "iso_region": "US-PA",
      "municipality": "Allentown",
      "scheduled_service": "yes",
      "gps_code": "KABE",
      "iata_code": "ABE",
      "local_code": "ABE",
      "wikipedia_link": "https://en.wikipedia.org/wiki/Lehigh_Valley_International_Airport"
    }
  ],
  "airport_freq_info": [
    {
      "_id": "68f2e8a2bb35bab1fe20aa51",
      "id": 60281,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "A/D",
      "description": "APP/DEP",
      "frequency_mhz": 118.2
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa52",
      "id": 60282,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "ATIS",
      "description": "ATIS",
      "frequency_mhz": 126.975
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa53",
      "id": 60283,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "CLD",
      "description": "CLNC DEL",
      "frequency_mhz": 124.05
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa54",
      "id": 60284,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "GND",
      "description": "GND",
      "frequency_mhz": 121.9
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa55",
      "id": 60285,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "RDO",
      "description": "WILLIAMSPORT RDO",
      "frequency_mhz": 117.5
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa56",
      "id": 60286,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "TWR",
      "description": "TWR",
      "frequency_mhz": 120.5
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa57",
      "id": 60287,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "UNIC",
      "description": "UNICOM",
      "frequency_mhz": 122.95
    }
  ]
}

3. Ceo dokument:
{
  "_id": "68f2e550bb35bab1febe2c98",
  "FlightDate": "2023-01-12 00:00:00",
  "Day_Of_Week": 4,
  "Airline": "Skywest Airlines Inc.",
  "Tail_Number": "N897SK",
  "Dep_Airport": "ABE",
  "Dep_CityName": "Allentown/Bethlehem/Easton, PA",
  "DepTime_label": "Afternoon",
  "Dep_Delay": 19,
  "Dep_Delay_Tag": 1,
  "Dep_Delay_Type": "Medium >15min",
  "Arr_Airport": "ATL",
  "Arr_CityName": "Atlanta, GA",
  "Arr_Delay": 20,
  "Arr_Delay_Type": "Medium >15min",
  "Flight_Duration": 146,
  "Distance_type": "Short Haul >1500Mi",
  "Delay_Carrier": 0,
  "Delay_Weather": 20,
  "Delay_NAS": 0,
  "Delay_Security": 0,
  "Delay_LastAircraft": 0,
  "Manufacturer": "CANADAIR REGIONAL JET",
  "Model": "CRJ",
  "Aicraft_age": 17,
  "airport_info": [
    {
      "_id": "68f2e8a7bb35bab1fe21024c",
      "id": 3356,
      "ident": "KABE",
      "type": "medium_airport",
      "name": "Lehigh Valley International Airport",
      "latitude_deg": 40.652099609375,
      "longitude_deg": -75.44080352783203,
      "elevation_ft": 393.0,
      "iso_country": "US",
      "iso_region": "US-PA",
      "municipality": "Allentown",
      "scheduled_service": "yes",
      "gps_code": "KABE",
      "iata_code": "ABE",
      "local_code": "ABE",
      "wikipedia_link": "https://en.wikipedia.org/wiki/Lehigh_Valley_International_Airport"
    }
  ],
  "airport_freq_info": [
    {
      "_id": "68f2e8a2bb35bab1fe20aa51",
      "id": 60281,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "A/D",
      "description": "APP/DEP",
      "frequency_mhz": 118.2
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa52",
      "id": 60282,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "ATIS",
      "description": "ATIS",
      "frequency_mhz": 126.975
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa53",
      "id": 60283,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "CLD",
      "description": "CLNC DEL",
      "frequency_mhz": 124.05
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa54",
      "id": 60284,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "GND",
      "description": "GND",
      "frequency_mhz": 121.9
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa55",
      "id": 60285,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "RDO",
      "description": "WILLIAMSPORT RDO",
      "frequency_mhz": 117.5
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa56",
      "id": 60286,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "TWR",
      "description": "TWR",
      "frequency_mhz": 120.5
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa57",
      "id": 60287,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "UNIC",
      "description": "UNICOM",
      "frequency_mhz": 122.95
    }
  ]
}

4. Ceo dokument:
{
  "_id": "68f2e550bb35bab1febdf4bd",
  "FlightDate": "2023-01-22 00:00:00",
  "Day_Of_Week": 7,
  "Airline": "Skywest Airlines Inc.",
  "Tail_Number": "N549CA",
  "Dep_Airport": "ABE",
  "Dep_CityName": "Allentown/Bethlehem/Easton, PA",
  "DepTime_label": "Afternoon",
  "Dep_Delay": 3,
  "Dep_Delay_Tag": 1,
  "Dep_Delay_Type": "Low <5min",
  "Arr_Airport": "ATL",
  "Arr_CityName": "Atlanta, GA",
  "Arr_Delay": 25,
  "Arr_Delay_Type": "Medium >15min",
  "Flight_Duration": 167,
  "Distance_type": "Short Haul >1500Mi",
  "Delay_Carrier": 0,
  "Delay_Weather": 25,
  "Delay_NAS": 0,
  "Delay_Security": 0,
  "Delay_LastAircraft": 0,
  "Manufacturer": "CANADAIR REGIONAL JET",
  "Model": "CRJ",
  "Aicraft_age": 16,
  "airport_info": [
    {
      "_id": "68f2e8a7bb35bab1fe21024c",
      "id": 3356,
      "ident": "KABE",
      "type": "medium_airport",
      "name": "Lehigh Valley International Airport",
      "latitude_deg": 40.652099609375,
      "longitude_deg": -75.44080352783203,
      "elevation_ft": 393.0,
      "iso_country": "US",
      "iso_region": "US-PA",
      "municipality": "Allentown",
      "scheduled_service": "yes",
      "gps_code": "KABE",
      "iata_code": "ABE",
      "local_code": "ABE",
      "wikipedia_link": "https://en.wikipedia.org/wiki/Lehigh_Valley_International_Airport"
    }
  ],
  "airport_freq_info": [
    {
      "_id": "68f2e8a2bb35bab1fe20aa51",
      "id": 60281,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "A/D",
      "description": "APP/DEP",
      "frequency_mhz": 118.2
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa52",
      "id": 60282,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "ATIS",
      "description": "ATIS",
      "frequency_mhz": 126.975
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa53",
      "id": 60283,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "CLD",
      "description": "CLNC DEL",
      "frequency_mhz": 124.05
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa54",
      "id": 60284,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "GND",
      "description": "GND",
      "frequency_mhz": 121.9
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa55",
      "id": 60285,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "RDO",
      "description": "WILLIAMSPORT RDO",
      "frequency_mhz": 117.5
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa56",
      "id": 60286,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "TWR",
      "description": "TWR",
      "frequency_mhz": 120.5
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa57",
      "id": 60287,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "UNIC",
      "description": "UNICOM",
      "frequency_mhz": 122.95
    }
  ]
}

5. Ceo dokument:
{
  "_id": "68f2e550bb35bab1febdee79",
  "FlightDate": "2023-01-23 00:00:00",
  "Day_Of_Week": 1,
  "Airline": "Skywest Airlines Inc.",
  "Tail_Number": "N800SK",
  "Dep_Airport": "ABE",
  "Dep_CityName": "Allentown/Bethlehem/Easton, PA",
  "DepTime_label": "Afternoon",
  "Dep_Delay": 34,
  "Dep_Delay_Tag": 1,
  "Dep_Delay_Type": "Medium >15min",
  "Arr_Airport": "ATL",
  "Arr_CityName": "Atlanta, GA",
  "Arr_Delay": 27,
  "Arr_Delay_Type": "Medium >15min",
  "Flight_Duration": 135,
  "Distance_type": "Short Haul >1500Mi",
  "Delay_Carrier": 0,
  "Delay_Weather": 27,
  "Delay_NAS": 0,
  "Delay_Security": 0,
  "Delay_LastAircraft": 0,
  "Manufacturer": "CANADAIR REGIONAL JET",
  "Model": "CRJ",
  "Aicraft_age": 18,
  "airport_info": [
    {
      "_id": "68f2e8a7bb35bab1fe21024c",
      "id": 3356,
      "ident": "KABE",
      "type": "medium_airport",
      "name": "Lehigh Valley International Airport",
      "latitude_deg": 40.652099609375,
      "longitude_deg": -75.44080352783203,
      "elevation_ft": 393.0,
      "iso_country": "US",
      "iso_region": "US-PA",
      "municipality": "Allentown",
      "scheduled_service": "yes",
      "gps_code": "KABE",
      "iata_code": "ABE",
      "local_code": "ABE",
      "wikipedia_link": "https://en.wikipedia.org/wiki/Lehigh_Valley_International_Airport"
    }
  ],
  "airport_freq_info": [
    {
      "_id": "68f2e8a2bb35bab1fe20aa51",
      "id": 60281,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "A/D",
      "description": "APP/DEP",
      "frequency_mhz": 118.2
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa52",
      "id": 60282,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "ATIS",
      "description": "ATIS",
      "frequency_mhz": 126.975
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa53",
      "id": 60283,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "CLD",
      "description": "CLNC DEL",
      "frequency_mhz": 124.05
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa54",
      "id": 60284,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "GND",
      "description": "GND",
      "frequency_mhz": 121.9
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa55",
      "id": 60285,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "RDO",
      "description": "WILLIAMSPORT RDO",
      "frequency_mhz": 117.5
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa56",
      "id": 60286,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "TWR",
      "description": "TWR",
      "frequency_mhz": 120.5
    },
    {
      "_id": "68f2e8a2bb35bab1fe20aa57",
      "id": 60287,
      "airport_ref": 3356,
      "airport_ident": "KABE",
      "type": "UNIC",
      "description": "UNICOM",
      "frequency_mhz": 122.95
    }
  ]
}
```

In [ ]:
test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "Airline" : { "$ne" : None },
            "Delay_Weather" : { "$gt" : 10, "$ne" : None }
        }
    },
    {
        "$lookup" : {
            "from" : "airports",
            "localField" : "Dep_Airport",
            "foreignField" : "iata_code",
            "as" : "airport_info"
        }
    },
    {
        "$lookup" : {
            "from" : "airport_frequencies",
            "localField" : "airport_info.ident",
            "foreignField" : "airport_ident",
            "as" : "airport_freq_info"
        }
    },
    {
        "$limit" : 5
    }
]

**Part 3:**
```
Test Results:
1. Ceo dokument:
{
  "_id": {
    "airline": "Allegiant Air",
    "airport": "PDX"
  },
  "weather_delay_count": 2,
  "total_weather_delay": 875,
  "average_weather_delay": 437.5,
  "airport_info": [],
  "airport_freq_info": []
}

2. Ceo dokument:
{
  "_id": {
    "airline": "Delta Air Lines Inc",
    "airport": "STL"
  },
  "weather_delay_count": 12,
  "total_weather_delay": 853,
  "average_weather_delay": 71.08333333333333,
  "airport_info": [],
  "airport_freq_info": []
}
```

In [ ]:
test_pipeline = [
    {
        "$match" : {
            "Dep_Airport" : { "$ne" : None },
            "FlightDate" : { "$ne" : None },
            "Dep_Delay" : { "$ne" : None },
            "Airline" : { "$ne" : None },
            "Delay_Weather" : { "$gt" : 10, "$ne" : None }
        }
    },
    {
        "$group" : {
            "_id" : {
                "airline" : "$Airline",
                "airport" : "$Dep_Airport"
            },
            "weather_delay_count" : { "$sum" : 1 },
            "total_weather_delay" : { "$sum" : "$Delay_Weather" },
            "average_weather_delay" : { "$avg" : "$Delay_Weather" }
        }
    },
    {
        "$lookup" : {
            "from" : "airports",
            "localField" : "Dep_Airport",
            "foreignField" : "iata_code",
            "as" : "airport_info"
        }
    },
    {
        "$lookup" : {
            "from" : "airport_frequencies",
            "localField" : "airport_info.ident",
            "foreignField" : "airport_ident",
            "as" : "airport_freq_info"
        }
    },
    {
        "$limit" : 5
    }
]